# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [53]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [54]:
# Initialize and constants

'''load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!") '''
    
MODEL = 'llama3.2'
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama") 

In [55]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [56]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [57]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links, do not invent urls, do not guess urls, do not modify urls.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [58]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links, do not invent urls, do not guess urls, do not modify urls.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-cod

In [60]:

def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [61]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linked in profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [79]:
from urllib.parse import urljoin, urldefrag

def normalize_url(base_url, link_url):
    """
    Convert relative URLs to absolute URLs and remove fragments.
    Reject non-string URLs safely.
    """

    if not isinstance(base_url, str):
        raise TypeError(f"base_url must be str, got {type(base_url)}")

    if not isinstance(link_url, str):
        return None

    link_url = link_url.strip()

    if not link_url:
        return None

    absolute_url = urljoin(base_url, link_url)
    clean_url, _fragment = urldefrag(absolute_url)

    return clean_url
    
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    actual_links = fetch_website_links(url)

    allowed_links = {
        normalize_url(url, link)
        for link in actual_links
    }

    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    cleaned_links = []

    for link in links.get("links", []):
        # Case 1: LLM returned a string URL
        if isinstance(link, str):
            link_type = "relevant page"
            link_url = link

        # Case 2: LLM returned the expected object format
        elif isinstance(link, dict):
            link_type = link.get("type", "relevant page")
            link_url = link.get("url")

        # Case 3: unexpected format
        else:
            print(f"Skipping invalid link format: {link}")
            continue

        if not link_url:
            print(f"Skipping link with missing URL: {link}")
            continue

        normalized_link_url = normalize_url(url, link_url)

        if normalized_link_url not in allowed_links:
            print(f"Skipping hallucinated link: {normalized_link_url}")
            continue

        cleaned_links.append({
            "type": link_type,
            "url": normalized_link_url
        })

    print(f"Found {len(cleaned_links)} relevant valid links")

    return {"links": cleaned_links}

In [80]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Skipping hallucinated link: https://edwarddonner.com/about-me-and-about-nebula
Found 3 relevant valid links


{'links': [{'type': 'company home page', 'url': 'https://edwarddonner.com'},
  {'type': 'careers page', 'url': 'https://news.ycombinator.com'},
  {'type': 'careers link', 'url': 'mailto:hello@mygroovydomain.com'}]}

In [81]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Skipping hallucinated link: https://docs.huggingface.co/
Skipping hallucinated link: https://www.huggingface.co/brand
Skipping hallucinated link: https://chat.huggingface.co/
Skipping hallucinated link: https://endpoints.huggingface.co/
Found 5 relevant valid links


{'links': [{'type': 'relevant page', 'url': 'https://huggingface.co/'},
  {'type': 'relevant page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'relevant page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'relevant page', 'url': 'https://status.huggingface.co/'},
  {'type': 'relevant page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links.get("links", []):
        if not isinstance(link, dict):
            print(f"Skipping non-dict link: {repr(link)}")
            continue

        link_type = link.get("type", "relevant page")
        link_url = link.get("url")

        print("FETCHING:", repr(link_url), type(link_url))

        if not isinstance(link_url, str):
            print(f"Skipping non-string URL: {repr(link_url)}")
            continue

        if not link_url.startswith(("http://", "https://")):
            print(f"Skipping invalid URL schema: {repr(link_url)}")
            continue

        result += f"\n\n### Link: {link_type}\n"
        result += fetch_website_contents(link_url)

    return result

In [83]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Skipping hallucinated link: https://blog.huggingface.co/
Found 12 relevant valid links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
9 days ago
•
1.68M
•
1.73k
zai-org/GLM-5.2
Updated
5 days ago
•
282k
•
3.58k
tencent/Hy3
Updated
1 day ago
•
121
•
449
baidu/Unlimited-OCR
Updated
5 days ago
•
1.08M
•
1.83k
Intern

In [84]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [85]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [86]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Skipping hallucinated link: https://huggingface.co
Skipping hallucinated link: https://huggingface.com
Skipping hallucinated link: https://discord.gg/huggingface
Skipping hallucinated link: https://docs.huggingface.co/transformers/
Found 6 relevant valid links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n9 days ago\n•\n1.68M\n•\n1.73k\nzai-org/GLM-5.2\nUpdated\n5 days ago\n•

In [87]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [88]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Skipping hallucinated link: https://docs.huggingface.co/transformers
Skipping hallucinated link: https://docs.huggingface.co/diffusers
Skipping hallucinated link: https://docs.huggingface.co/safetensors
Skipping hallucinated link: https://huggingface_hub.com/
Found 4 relevant valid links


# Hugging Face: Accelerating Artificial Intelligence Development
=====================================

**About Us**
---------------

Hugging Face is the AI community building the future. Our platform is designed to facilitate collaboration, innovation, and adoption of Artificial Intelligence (AI) across various industries.

**Our Mission**
-----------------

To empower researchers, developers, and businesses with the tools and resources necessary to harness the power of AI, democratizing access to cutting-edge machine learning models, datasets, and applications.

**What We Offer**
-----------------

*   **Unlimited Public Models and Datasets**: Host, collaborate on, and discover millions of high-quality models, datasets, and applications.
*   **AI Code Creation**: Leverage our AI-powered tools like GitHub Copilot to write better code and accelerate development.
*   **Collaboration Platform**: Join the community and engage with other researchers, developers, and experts in the field.
*   **Explore All Modalities**: Discover text, image, and more modalities for AI research and development.

### Space

## Hugging Face's vision is to create a hub of innovation and knowledge-sharing around Artificial Intelligence (AI).

## We aim to enable collaboration among researchers, developers, business, and academics by developing innovative tools including but not limited to machine learning models.

## [Click here](#) for more information about **Hugging Face**.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [89]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [90]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Skipping hallucinated link: https://blog.huggingface.co/
Skipping hallucinated link: https://docs.huggingface.co/
Skipping hallucinated link: https://papers.huggingface.co/
Skipping hallucinated link: https://discord.huggingface.co/
Found 8 relevant valid links


InvalidSchema: No connection adapters were found for 'mailto:press@huggingface.co'

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>